In [2]:
# ==========================================
# 爆速・常に綺麗・位置ズレなしの血管中心線抽出スクリプト（完全版）
# ==========================================

import os
import json
import numpy as np
import SimpleITK as sitk
from scipy.ndimage import distance_transform_edt, binary_closing
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from skan import Skeleton

# ------------------------------------------
# 1. パスおよびパラメータ設定
# ------------------------------------------
DICOM_DIR = '/Users/tkimura/Desktop/Image data/2-ope_cases/LTxD/DICOM/4882127/DICOM/DICOM-A/ScalarVolume_22'
OUTPUT_JSON_PATH = 'vessel_centerline_dual.json'

# CT閾値 (HU)
TH_MIN = 150
TH_MAX = 1000

# フィルタリング設定
MIN_VOXEL_SIZE = 500   # このボクセル数未満の孤立ノイズを破棄
MIN_PATH_LENGTH = 15   # この画素数未満の短いぶつ切り枝を破棄

# ------------------------------------------
# 2. DICOM データの読み込み
# ------------------------------------------
print("📂 DICOM データを読み込み中...")
reader = sitk.ImageSeriesReader()
dicom_names = reader.GetGDCMSeriesFileNames(DICOM_DIR)
reader.SetFileNames(dicom_names)
image = reader.Execute()

data_np = sitk.GetArrayFromImage(image)
spacing = np.array(image.GetSpacing())
origin = np.array(image.GetOrigin())

print(f"  - Volume Shape : {data_np.shape}")
print(f"  - Spacing (mm) : {spacing}")
print(f"  - Origin (mm)  : {origin}")

# ------------------------------------------
# 3. 領域抽出（二値化）＋ 3D Closing による虫食い補正
# ------------------------------------------
print("✂️ 血管領域の二値化 処理中...")
binary_raw = (data_np >= TH_MIN) & (data_np <= TH_MAX)

print("🔄 血管のぶつ切り・虫食い隙間を自動接着中（3D Closing）...")
# 3×3×3の構造要素で半径2〜3ピクセルの空隙を埋めて1本の管に接続
struct_elem = np.ones((3, 3, 3), dtype=bool)
binary_vessel = binary_closing(binary_raw, structure=struct_elem).astype(np.uint8)

# ------------------------------------------
# 4. ノイズ除去（主要成分の抽出） ＆ 爆速スケルトン化
# ------------------------------------------
print("🧹 細かい孤立ノイズを除去中...")
labeled_vessel, num_features = label(binary_vessel, return_num=True)

binary_vessel_clean = np.zeros_like(binary_vessel)
for prop in regionprops(labeled_vessel):
    if prop.area >= MIN_VOXEL_SIZE:
        binary_vessel_clean[labeled_vessel == prop.label] = 1

print(f"  - {num_features} 個のコンポーネントから主要領域へ絞り込み完了")

print("⏳ 距離マップ計算 ＆ スケルトン化 処理中...")
dist_map = distance_transform_edt(binary_vessel_clean)
skeleton = skeletonize(binary_vessel_clean)

# ------------------------------------------
# 5. skan による中心線抽出 ＆ 主要パスの抽出
# ------------------------------------------
print("🔗 血管パスの解析 ＆ 再統合中...")
skel_obj = Skeleton(skeleton)
all_segments = []

num_edges = skel_obj.paths.shape[0]

for edge_idx in range(num_edges):
    path_pixels = skel_obj.path_coordinates(edge_idx)
    
    # 短いノイズ枝を除外
    if len(path_pixels) < MIN_PATH_LENGTH:
        continue
        
    # (x, y, z) の順に並び替え
    pts = path_pixels[:, [2, 1, 0]].tolist()
    all_segments.append(pts)

print(f"🎉 抽出完了: {num_edges} 本の細切れデータから、繋がった {len(all_segments)} 本の主要血管パスへ集約しました！")

# ------------------------------------------
# 6. Blender / Unity 互換座標で JSON 保存
# ------------------------------------------
vol_center_idx = np.array(data_np.shape[::-1]) / 2.0
vol_center_mm = origin + vol_center_idx * spacing

output_data = {
    "metadata": {
        "unit": "mm", 
        "vessel_count": len(all_segments)
    },
    "polylines": []
}

for idx, seg in enumerate(all_segments):
    blender_nodes = []
    unity_nodes = []
    
    for pt in seg:
        pt_np = np.array(pt)
        phys_mm = origin + pt_np * spacing
        centered_mm = phys_mm - vol_center_mm
        
        radius_vox = dist_map[int(pt[2]), int(pt[1]), int(pt[0])]
        radius_mm = radius_vox * spacing[0]
        
        blender_nodes.append({
            "pos": centered_mm.tolist(),
            "r": float(radius_mm)
        })
        
        unity_nodes.append({
            "pos": [centered_mm[0], centered_mm[2], centered_mm[1]],
            "r": float(radius_mm)
        })
        
    output_data["polylines"].append({
        "id": idx,
        "blender": blender_nodes,
        "unity": unity_nodes
    })

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

print(f"💾 [SUCCESS] 正常に保存されました -> {os.path.abspath(OUTPUT_JSON_PATH)}")

📂 DICOM データを読み込み中...
  - Volume Shape : (111, 512, 512)
  - Spacing (mm) : [0.625 0.625 2.   ]
  - Origin (mm)  : [ -159.843  -178.593 -1204.   ]
✂️ 血管領域の二値化 処理中...
🔄 血管のぶつ切り・虫食い隙間を自動接着中（3D Closing）...
🧹 細かい孤立ノイズを除去中...
  - 902 個のコンポーネントから主要領域へ絞り込み完了
⏳ 距離マップ計算 ＆ スケルトン化 処理中...
🔗 血管パスの解析 ＆ 再統合中...
🎉 抽出完了: 19533 本の細切れデータから、繋がった 733 本の主要血管パスへ集約しました！
💾 [SUCCESS] 正常に保存されました -> /Users/tkimura/Downloads/vessel_centerline_dual.json
